# FoodVision AI - YOLO11n GPU Training Pipeline (Google Colab / Kaggle T4)

**Card #20 (`c5rVF4w5`) & Card #13 (`t4kdIXx5`) - TV1 (Minh Thanh)**

Quy trinh huan luyen chuan tren GPU T4:
1. Kiem tra GPU CUDA (`Tesla T4` / `L4`) va cai dat `ultralytics`.
2. Giai nen goi `food-ingredients-v1.zip` (da chuan hoa 70/15/15 theo `scene_id` tu `scripts/prepare_yolo_dataset.py`).
3. Huan luyen `yolo11n.pt` (`imgsz=640`, `epochs=100`, `patience=20`, `mosaic=0.7`, `mixup=0.1`, `seed=42`).
4. Kiem dinh tren tap `val` (giu dong bang tap `test` cho Giai doan 5 `NFR-11`), tinh ma bam `SHA-256` va tai `ingredients_best.pt` ve may local.
5. Tren may local, chay 1 lenh duy nhat de dong bo toan bo he thong:
   `python scripts/train_yolo_v1.py --import-weights path/to/ingredients_best.pt`

In [ ]:
!pip install -q ultralytics pyyaml
import torch
import ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Vui long bat Runtime -> Change runtime type -> T4 GPU!'

In [ ]:
# Cach 1: Mount Google Drive chua tep nen datasets/processed/food-ingredients-v1.zip
from pathlib import Path
import zipfile

ZIP_PATH = Path('/content/drive/MyDrive/food-ingredients-v1.zip')
EXTRACT_DIR = Path('/content/datasets/processed/food-ingredients-v1')

if ZIP_PATH.exists():
    EXTRACT_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Extracted dataset to:', EXTRACT_DIR)
else:
    print('Upload food-ingredients-v1.zip to Google Drive or /content/ before running next cell.')

In [ ]:
# Cap nhat duong dan tuyet doi trong food_ingredients_v1.yaml cho moi truong Colab/Kaggle
import yaml

yaml_path = EXTRACT_DIR / 'food_ingredients_v1.yaml'
cfg = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
cfg['path'] = str(EXTRACT_DIR)
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(yaml_path.read_text(encoding='utf-8'))

In [ ]:
# Huan luyen YOLO11n tren tap train/val (TUYET DOI KHONG DUNG split='test' de chinh tham so - NFR-11)
import hashlib
import shutil
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data=str(yaml_path),
    imgsz=640,
    epochs=100,
    patience=20,
    batch=32,
    mosaic=0.7,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    seed=42,
    deterministic=True,
    device=0,
    project='/content/runs/detect',
    name='foodvision_yolo_v1',
    exist_ok=True,
)

best_pt = Path('/content/runs/detect/foodvision_yolo_v1/weights/best.pt')
export_pt = Path('/content/ingredients_best.pt')
shutil.copyfile(best_pt, export_pt)
with open(export_pt, 'rb') as fh:
    sha256_hex = hashlib.file_digest(fh, 'sha256').hexdigest()

print('Exported:', export_pt, export_pt.stat().st_size, 'bytes')
print('SHA-256 (64 hex):', sha256_hex)

In [ ]:
# Kiem tra chi so tren tap validation (khong dung tap test)
trained = YOLO(str(export_pt))
val_metrics = trained.val(data=str(yaml_path), split='val')
print('Validation mAP@50:', round(float(val_metrics.box.map50), 4))
print('Validation mAP@50:95:', round(float(val_metrics.box.map), 4))